# Error Analysis: Road Accident Prediction

สมุดบันทึกนี้อ่านผลทำนายจาก artifact ที่สร้างโดย pipeline หลัก และวิเคราะห์ error ด้วย schema ปัจจุบันของโปรเจกต์

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError("ไม่พบโฟลเดอร์ src กรุณาเปิด notebook จาก project root หรือโฟลเดอร์ notebooks")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("bmh")
plt.rcParams["figure.figsize"] = (10, 4)

from src.utils.config import PREDICTIONS_SAMPLE_CSV


In [ ]:
pred_path = PREDICTIONS_SAMPLE_CSV if PREDICTIONS_SAMPLE_CSV.exists() else PROJECT_ROOT / "artifacts" / "predictions_sample.csv"
pred_df = pd.read_csv(pred_path)

pred_col = "predicted" if "predicted" in pred_df.columns else "prediction_label"
if "actual" not in pred_df.columns:
    raise KeyError("predictions_sample.csv ต้องมีคอลัมน์ actual")

pred_df["actual"] = pd.to_numeric(pred_df["actual"], errors="coerce")
pred_df[pred_col] = pd.to_numeric(pred_df[pred_col], errors="coerce")
if "abs_error" not in pred_df.columns:
    pred_df["abs_error"] = (pred_df["actual"] - pred_df[pred_col]).abs()
pred_df["signed_error"] = pred_df[pred_col] - pred_df["actual"]

print(f"loaded: {pred_path}")
print(f"rows: {len(pred_df):,}")
pred_df.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(pred_df["abs_error"].dropna(), bins=30, color="#ef8354", edgecolor="white", alpha=0.9)
axes[0].set_title("Distribution of Absolute Error")
axes[0].set_xlabel("abs_error")

axes[1].scatter(pred_df["actual"], pred_df[pred_col], alpha=0.35, color="#2a6f97")
limits = [
    min(pred_df["actual"].min(), pred_df[pred_col].min()),
    max(pred_df["actual"].max(), pred_df[pred_col].max()),
]
axes[1].plot(limits, limits, linestyle="--", color="black", linewidth=1)
axes[1].set_title("Actual vs Predicted")
axes[1].set_xlabel("actual")
axes[1].set_ylabel(pred_col)

plt.tight_layout()
plt.show()


In [ ]:
columns_to_show = [
    c
    for c in [
        "จังหวัด", "สภาพอากาศ", "ลักษณะการเกิดเหตุ", "มูลเหตุสันนิษฐาน",
        "บริเวณที่เกิดเหตุ", "actual", pred_col, "abs_error", "signed_error"
    ]
    if c in pred_df.columns
]

worst_cases = pred_df.sort_values("abs_error", ascending=False)[columns_to_show].head(10)
worst_cases


In [ ]:
if "จังหวัด" in pred_df.columns:
    province_error = (
        pred_df.groupby("จังหวัด")["abs_error"]
        .agg(count="size", mean="mean", median="median")
        .query("count >= 20")
        .sort_values("mean", ascending=False)
        .head(10)
    )
    province_error
else:
    pred_df["abs_error"].describe()


## แนวทางใช้ผลลัพธ์

- ดู `worst_cases` เพื่อหา pattern ของกรณีที่โมเดลพลาดมากที่สุด
- ดูตาราง `province_error` เพื่อหาจังหวัดที่อาจต้องเพิ่มข้อมูลหรือฟีเจอร์เฉพาะพื้นที่
- หาก artifact ยังไม่อัปเดต ให้รัน notebook modeling หรือ `python -m src.model.train_pycaret` ก่อน